# GWM-E Link Prediction Training (Cross-Attention - Google Colab)

Train the cross-attention architecture on Google Colab with Google Drive storage.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project directory in Google Drive
PROJECT_ROOT = '/content/drive/MyDrive/NLP-research/code/GWM/link-prediction/'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/cora', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/trained/cross-attn/cora', exist_ok=True)

print(f"✓ Google Drive mounted")
print(f"✓ Project directory: {PROJECT_ROOT}")

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.3
!pip install -q transformers>=4.35.0 accelerate sentencepiece huggingface-hub tqdm matplotlib

print("✓ All dependencies installed")

## 3. Check GPU and Environment

In [ ]:
import torch
import sys

print("="*70)
print(" "*20 + "COLAB ENVIRONMENT")
print("="*70)
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️  WARNING: No GPU detected! Training will be very slow.")
    print("   Go to Runtime > Change runtime type > Hardware accelerator > GPU")

print("="*70)

## 4. Copy Data and Training Files from GitHub

All data and code will be loaded from your GitHub repository. Google Drive is only used for saving trained models.

In [ ]:
required_files = ['model.py', 'dataset.py', 'inference.py', 'train.py', 'utils.py']
data_files = [
    'cora_train_link_data.jsonl',
    'train_edge_embeddings.pt',
    'cora_val_link_data.jsonl',
    'val_edge_embeddings.pt',
    'cora_test_link_data.jsonl',
    'test_edge_embeddings.pt'
]

print("="*70)
print("Cloning GitHub repository...")
print("="*70)

# Clone your GitHub repo
GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
BRANCH = "main"

!git clone {GITHUB_REPO} /content/gwm
%cd /content/gwm
!git checkout {BRANCH}
!git pull
%cd /content

# Copy training files from cross-attn folder to working directory
repo_path = "/content/gwm/gwm/link-prediction/cross-attn"
data_path = "/content/gwm/data/cora/train"  # Adjust this path to match your repo structure

print(f"\nCopying cross-attention files from {repo_path}...")
for file in required_files:
    !cp {repo_path}/{file} /content/
    print(f"✓ Copied {file}")

print(f"\nCopying data files from {data_path}...")
for file in data_files:
    !cp {data_path}/{file} /content/
    print(f"✓ Copied {file}")

# Verify files exist
import os
missing_files = [f for f in required_files + data_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready")
    print(f"✓ Code files: {required_files}")
    print(f"✓ Data files: {data_files}")
    print("✓ Using CROSS-ATTENTION architecture")

## 5. Configuration - Parameter Grid

Define multiple parameter sets to train. Each configuration will be trained sequentially.

**Note:** Cross-attention typically needs lower learning rates than baseline due to higher model complexity.

In [ ]:
# ==============================================================================
# PARAMETER GRID - Multiple Training Configurations
# ==============================================================================
# Each config will be trained sequentially and saved to a separate folder

PARAM_GRID = [
    {
        'name': 'crossattn_lower_lr',
        'learning_rate': 1e-5,  # Lower LR for cross-attention
        'weight_decay': 0.1,
        'dropout': 0.1,
        'batch_size': 4,
        'gradient_accumulation': 8,
        'early_stopping_patience': 10,
    },
    {
        'name': 'crossattn_very_low_lr',
        'learning_rate': 5e-6,  # Very conservative LR
        'weight_decay': 0.1,
        'dropout': 0.1,
        'batch_size': 4,
        'gradient_accumulation': 8,
        'early_stopping_patience': 10,
    },
    {
        'name': 'crossattn_high_reg',
        'learning_rate': 1e-5,
        'weight_decay': 0.2,  # Stronger regularization
        'dropout': 0.3,
        'batch_size': 4,
        'gradient_accumulation': 8,
        'early_stopping_patience': 10,
    },
]

# ==============================================================================
# SHARED CONFIGURATION (Same for all parameter sets)
# ==============================================================================
# Data paths (loaded from /content/ - from GitHub)
DATA_DIR = '/content'

TRAIN_JSONL = f'{DATA_DIR}/cora_train_link_data.jsonl'
TRAIN_EMBEDDING = f'{DATA_DIR}/train_edge_embeddings.pt'
VAL_JSONL = f'{DATA_DIR}/cora_val_link_data.jsonl'
VAL_EMBEDDING = f'{DATA_DIR}/val_edge_embeddings.pt'
TEST_JSONL = f'{DATA_DIR}/cora_test_link_data.jsonl'
TEST_EMBEDDING = f'{DATA_DIR}/test_edge_embeddings.pt'

# Google Drive output directory (for checkpoints only)
PROJECT_ROOT = '/content/drive/MyDrive/NLP-research/code/GWM/link-prediction/'

# Model configuration (same for all runs)
LLAMA_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'
GRAPH_EMBEDDING_DIM = 768
PROJECTOR_HIDDEN_DIM = 3072  # Cross-attention architecture
NUM_HOPS = 4

# Training settings (same for all runs)
NUM_EPOCHS = 20
WARMUP_STEPS = 100  # Increased warmup for cross-attention
MAX_GRAD_NORM = 1.0
USE_FP16 = True
NUM_WORKERS = 4

# Display configuration
print("="*70)
print(" "*10 + "GWM CROSS-ATTENTION - MULTI-CONFIG TRAINING")
print("="*70)
print(f"\n📊 Architecture: CROSS-ATTENTION")
print(f"   Projector hidden dim: {PROJECTOR_HIDDEN_DIM}")
print(f"\n📁 Data Source: GitHub Repository")
print(f"   Files loaded to: {DATA_DIR}")
print(f"\n💾 Checkpoint Storage: Google Drive")
print(f"   Base directory: {PROJECT_ROOT}/trained/cross-attn/cora/")
print(f"\n🎯 Training Configurations: {len(PARAM_GRID)} sets")
print(f"   ⚠️  All configs use LOWER learning rates than baseline")
for i, config in enumerate(PARAM_GRID, 1):
    print(f"\n   {i}. {config['name']}")
    print(f"      LR={config['learning_rate']:.1e}, WD={config['weight_decay']}, Dropout={config['dropout']}")
    print(f"      Batch={config['batch_size']}, GradAccum={config['gradient_accumulation']}")
print(f"\n⏱️  Estimated time: ~{len(PARAM_GRID) * 12} hours on A100 ({len(PARAM_GRID)} configs × ~12h each)")
print("="*70)

## 6. Authenticate with Hugging Face

In [ ]:
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    !huggingface-cli login --token {HF_TOKEN}
    print("✓ Logged in to Hugging Face")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n⚠️  Please add HF_TOKEN to Colab Secrets:")
    print("   1. Click 🔑 Secrets in left sidebar")
    print("   2. Add secret: HF_TOKEN = your_token")
    print("   3. Enable notebook access")
    print("   4. Get token from: https://huggingface.co/settings/tokens")

## 7. Train All Configurations

Train the model with each parameter set. Results are saved to Google Drive with descriptive folder names.

In [ ]:
import time
from datetime import datetime
import json

# Store results from all runs
all_results = []

print("="*70)
print(" "*15 + "STARTING MULTI-CONFIG TRAINING")
print("="*70)
print(f"Total configurations: {len(PARAM_GRID)}")
print(f"Estimated total time: ~{len(PARAM_GRID) * 12} hours")
print("="*70 + "\n")

for config_idx, param_config in enumerate(PARAM_GRID, 1):
    print("\n" + "="*70)
    print(f" CONFIG {config_idx}/{len(PARAM_GRID)}: {param_config['name']}")
    print("="*70)
    
    # Create output directory for this config
    config_output_dir = f"{PROJECT_ROOT}/trained/cross-attn/cora/{param_config['name']}"
    
    # Build training command
    cmd = f"""python train.py \\
        --train_jsonl {TRAIN_JSONL} \\
        --train_embedding {TRAIN_EMBEDDING} \\
        --val_jsonl {VAL_JSONL} \\
        --val_embedding {VAL_EMBEDDING} \\
        --test_jsonl {TEST_JSONL} \\
        --test_embedding {TEST_EMBEDDING} \\
        --output_dir {config_output_dir} \\
        --llama_model {LLAMA_MODEL} \\
        --graph_embedding_dim {GRAPH_EMBEDDING_DIM} \\
        --projector_hidden_dim {PROJECTOR_HIDDEN_DIM} \\
        --num_hops {NUM_HOPS} \\
        --dropout {param_config['dropout']} \\
        --batch_size {param_config['batch_size']} \\
        --gradient_accumulation_steps {param_config['gradient_accumulation']} \\
        --lr {param_config['learning_rate']} \\
        --weight_decay {param_config['weight_decay']} \\
        --epochs {NUM_EPOCHS} \\
        --warmup_steps {WARMUP_STEPS} \\
        --max_grad_norm {MAX_GRAD_NORM} \\
        --early_stopping_patience {param_config['early_stopping_patience']} \\
        --num_workers {NUM_WORKERS}"""
    
    if USE_FP16:
        cmd += " \\\n        --use_fp16"
    
    print(f"\n🚀 Training with:")
    print(f"   Learning rate: {param_config['learning_rate']:.1e}")
    print(f"   Weight decay: {param_config['weight_decay']}")
    print(f"   Dropout: {param_config['dropout']}")
    print(f"   Batch size: {param_config['batch_size']}")
    print(f"   Gradient accumulation: {param_config['gradient_accumulation']}")
    print(f"   Output: {config_output_dir}")
    print(f"\n{cmd}\n")
    
    # Execute training
    start_time = time.time()
    !{cmd}
    elapsed_time = time.time() - start_time
    
    # Load results
    results_path = f"{config_output_dir}/final_results.json"
    try:
        with open(results_path, 'r') as f:
            results = json.load(f)
        
        all_results.append({
            'config_name': param_config['name'],
            'parameters': param_config,
            'best_val_accuracy': results['best_val_accuracy'],
            'test_accuracy': results['test_accuracy'],
            'best_epoch': results['best_epoch'],
            'total_epochs': results['total_epochs'],
            'training_time': elapsed_time,
        })
        
        print(f"\n✓ Config {config_idx} completed!")
        print(f"  Val Acc: {results['best_val_accuracy']*100:.2f}%")
        print(f"  Test Acc: {results['test_accuracy']*100:.2f}%")
        print(f"  Time: {elapsed_time/3600:.1f}h")
    except Exception as e:
        print(f"\n❌ Failed to load results: {e}")
        all_results.append({
            'config_name': param_config['name'],
            'parameters': param_config,
            'error': str(e),
            'training_time': elapsed_time,
        })
    
    print("="*70)

print("\n" + "="*70)
print(" "*15 + "ALL CONFIGURATIONS COMPLETED")
print("="*70)
print(f"\n📊 Results Summary:\n")
for i, result in enumerate(all_results, 1):
    if 'test_accuracy' in result:
        print(f"{i}. {result['config_name']:25s} | Val: {result['best_val_accuracy']*100:5.2f}% | Test: {result['test_accuracy']*100:5.2f}% | Time: {result['training_time']/3600:.1f}h")
    else:
        print(f"{i}. {result['config_name']:25s} | ❌ FAILED")

# Save comparison results
comparison_path = f"{PROJECT_ROOT}/trained/cross-attn/cora/all_configs_comparison.json"
with open(comparison_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n✓ Comparison saved to: {comparison_path}")
print("="*70)

## 8. Visualize Comparison Across All Configurations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if all_results and 'test_accuracy' in all_results[0]:
    # Extract metrics
    config_names = [r['config_name'].replace('crossattn_', '') for r in all_results if 'test_accuracy' in r]
    val_accs = [r['best_val_accuracy'] * 100 for r in all_results if 'test_accuracy' in r]
    test_accs = [r['test_accuracy'] * 100 for r in all_results if 'test_accuracy' in r]
    learning_rates = [r['parameters']['learning_rate'] for r in all_results if 'test_accuracy' in r]
    
    # Create comparison plots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Test Accuracy Comparison
    x = np.arange(len(config_names))
    ax1.bar(x, test_accs, color=['#2ecc71' if acc > 85 else '#e74c3c' for acc in test_accs])
    ax1.set_xlabel('Configuration')
    ax1.set_ylabel('Test Accuracy (%)')
    ax1.set_title('Test Accuracy Across Configurations')
    ax1.set_xticks(x)
    ax1.set_xticklabels(config_names, rotation=45, ha='right')
    ax1.axhline(y=85.98, color='blue', linestyle='--', label='Baseline (85.98%)', linewidth=2)
    ax1.axhline(y=94.31, color='green', linestyle='--', label='Paper Target (94.31%)', linewidth=2)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. Val vs Test Accuracy
    ax2.scatter(val_accs, test_accs, s=100, alpha=0.7)
    for i, name in enumerate(config_names):
        ax2.annotate(name, (val_accs[i], test_accs[i]), fontsize=8, alpha=0.7)
    ax2.plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Perfect correlation')
    ax2.set_xlabel('Best Validation Accuracy (%)')
    ax2.set_ylabel('Test Accuracy (%)')
    ax2.set_title('Validation vs Test Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Learning Rate vs Accuracy
    ax3.scatter(learning_rates, test_accs, s=100, alpha=0.7)
    for i, name in enumerate(config_names):
        ax3.annotate(name, (learning_rates[i], test_accs[i]), fontsize=8, alpha=0.7)
    ax3.set_xlabel('Learning Rate')
    ax3.set_ylabel('Test Accuracy (%)')
    ax3.set_title('Learning Rate vs Test Accuracy')
    ax3.set_xscale('log')
    ax3.axhline(y=85.98, color='blue', linestyle='--', alpha=0.5, label='Baseline')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Metrics Table
    ax4.axis('off')
    table_data = []
    for r in all_results:
        if 'test_accuracy' in r:
            table_data.append([
                r['config_name'].replace('crossattn_', ''),
                f"{r['parameters']['learning_rate']:.1e}",
                f"{r['test_accuracy']*100:.2f}%",
                f"{r['training_time']/3600:.1f}h"
            ])
    
    table = ax4.table(
        cellText=table_data,
        colLabels=['Config', 'LR', 'Test Acc', 'Time'],
        cellLoc='left',
        loc='center',
        colWidths=[0.35, 0.2, 0.25, 0.2]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    ax4.set_title('Summary Table', fontsize=12, pad=20)
    
    plt.tight_layout()
    
    # Save plot
    plot_path = f"{PROJECT_ROOT}/trained/cross-attn/cora/all_configs_comparison.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Comparison plot saved to: {plot_path}")
    
    # Find best configuration
    best_config = max(all_results, key=lambda x: x.get('test_accuracy', 0) if 'test_accuracy' in x else 0)
    print(f"\n🏆 BEST CONFIGURATION: {best_config['config_name']}")
    print(f"   Test Accuracy: {best_config['test_accuracy']*100:.2f}%")
    print(f"   Learning Rate: {best_config['parameters']['learning_rate']:.1e}")
    print(f"   Dropout: {best_config['parameters']['dropout']}")
    print(f"   Weight Decay: {best_config['parameters']['weight_decay']}")
else:
    print("❌ No results to visualize. Make sure training completed successfully.")